# 02. ConversationTokenBufferMemory → `trim_messages` (+ `@before_model` 미들웨어)

| legacy | LangGraph / LangChain v1 |
|---|---|
| `ConversationTokenBufferMemory(llm=llm, max_token_limit=150)` | `trim_messages(messages, max_tokens=150, token_counter=llm, strategy="last")` |
| 메모리 내부에서 자동으로 오래된 메시지 제거 | 그래프 노드 안에서 호출 / `create_agent` 에서는 `@before_model` 미들웨어로 호출 |

> `trim_messages` 는 `langchain_core` 의 함수로 **legacy 시절에도 있던 기능 그대로** 입니다. 달라진 것은 "어디서 호출하느냐" 뿐입니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

dialog = [
    ("안녕하세요, 저는 최근에 여러분 회사의 공작 기계를 구매했습니다. 설치 방법을 알려주실 수 있나요?",
     "안녕하세요! 구매해 주셔서 감사합니다. 해당 기계 모델 번호를 알려주시겠어요?"),
    ("네, 모델 번호는 XG-200입니다.",
     "감사합니다. XG-200 모델의 설치 안내를 도와드리겠습니다. 먼저, 설치할 장소의 전원 공급 상태를 확인해주세요. 기계는 220V 전원이 필요합니다."),
    ("전원은 확인했습니다. 다음 단계는 무엇인가요?",
     "좋습니다. 다음으로, 기계를 평평하고 안정적인 바닥에 배치해 주세요. 이후, 제공된 사용자 매뉴얼에 따라 케이블 연결을 진행해 주시기 바랍니다."),
    ("연결은 어떻게 하나요?",
     "매뉴얼의 5페이지를 참조해 주세요. 케이블 연결에 관한 상세한 지침이 있습니다. 이 과정에서 어려움이 있으시면 추가적으로 도와드리겠습니다."),
    ("설치가 완료되면 어떻게 해야 하나요?",
     "설치가 완료되면, 전원을 켜고 초기 구동 테스트를 진행해 주시기 바랍니다. 테스트 절차는 매뉴얼의 10페이지에 설명되어 있습니다."),
    ("감사합니다, 도움이 많이 되었어요!",
     "언제든지 도와드릴 준비가 되어 있습니다. 추가적인 질문이나 지원이 필요하시면 언제든지 문의해 주세요. 좋은 하루 되세요!"),
]
history = [m for h, a in dialog for m in (HumanMessage(h), AIMessage(a))]
print("전체 메시지 수:", len(history), "/ 토큰 수:", llm.get_num_tokens_from_messages(history))

전체 메시지 수: 12 / 토큰 수: 352


## 1. legacy `load_memory_variables` 결과와 같은 것: 최근 150 토큰만 남기기

In [3]:
trimmed = trim_messages(
    history,
    max_tokens=150,
    token_counter=llm,      # legacy 의 llm= 인자와 같은 역할 (모델 토크나이저로 계산)
    strategy="last",        # 최근 메시지를 남긴다
    start_on="human",       # Human 메시지로 시작하도록 정리
)
print("남은 메시지 수:", len(trimmed), "/ 토큰 수:", llm.get_num_tokens_from_messages(trimmed))
for m in trimmed:
    print(type(m).__name__, "|", m.content)

남은 메시지 수: 4 / 토큰 수: 113
HumanMessage | 설치가 완료되면 어떻게 해야 하나요?
AIMessage | 설치가 완료되면, 전원을 켜고 초기 구동 테스트를 진행해 주시기 바랍니다. 테스트 절차는 매뉴얼의 10페이지에 설명되어 있습니다.
HumanMessage | 감사합니다, 도움이 많이 되었어요!
AIMessage | 언제든지 도와드릴 준비가 되어 있습니다. 추가적인 질문이나 지원이 필요하시면 언제든지 문의해 주세요. 좋은 하루 되세요!


> `token_counter=llm` 대신 `from langchain_core.messages.utils import count_tokens_approximately` 를 쓰면 모델 호출/토크나이저 없이 빠르게 근사 계산합니다. (Ollama 같은 로컬 모델에서 유용)

## 2. `create_agent` 에서 사용: `@before_model` 미들웨어
모델 호출 직전에 state 의 메시지를 잘라낸 결과로 **교체**합니다. `RemoveMessage(id=REMOVE_ALL_MESSAGES)` 는 "기존 메시지 전부 삭제" 명령입니다.

In [4]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime


@before_model
def keep_last_150_tokens(state: AgentState, runtime: Runtime):
    trimmed = trim_messages(
        state["messages"], max_tokens=150, token_counter=llm,
        strategy="last", start_on="human",
    )
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *trimmed]}


agent = create_agent(
    model=llm,
    tools=[],
    middleware=[keep_last_150_tokens],
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "machine-1"}}
out = agent.invoke(
    {"messages": history + [HumanMessage("제가 처음에 알려드린 모델 번호가 뭐였죠? 모르면 모른다고 답해주세요.")]},
    cfg,
)
print(out["messages"][-1].content)
print("state 에 남은 메시지 수:", len(agent.get_state(cfg).values["messages"]))

죄송하지만, 이전 대화에서 모델 번호를 언급하신 적이 없어서 기억하고 있지 않습니다. 모델 번호를 알려주시면 더 도움을 드릴 수 있습니다!
state 에 남은 메시지 수: 6


모델 번호(XG-200)는 150 토큰 밖으로 밀려나 잘렸기 때문에 기억하지 못합니다. legacy `ConversationTokenBufferMemory` 와 같은 동작입니다.

## 3. state 는 그대로 두고 LLM에 보낼 때만 자르려면
직접 만든 그래프라면 노드 안에서 `trim_messages` 결과를 `llm.invoke()` 에만 넘기고 state 는 건드리지 않으면 됩니다. (01번 노트북의 "방법 A" 참고)

In [5]:
from langgraph.graph import StateGraph, MessagesState, START


def chatbot(state: MessagesState):
    recent = trim_messages(state["messages"], max_tokens=150, token_counter=llm,
                           strategy="last", start_on="human")
    return {"messages": [llm.invoke(recent)]}


graph = StateGraph(MessagesState).add_node("chatbot", chatbot).add_edge(START, "chatbot").compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "machine-2"}}
graph.invoke({"messages": history + [HumanMessage("테스트 절차는 몇 페이지에 있죠?")]}, cfg)
print(graph.get_state(cfg).values["messages"][-1].content)
print("state 에 남은 메시지 수:", len(graph.get_state(cfg).values["messages"]), "(전체 보존)")

테스트 절차는 매뉴얼의 10페이지에 설명되어 있습니다. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!
state 에 남은 메시지 수: 14 (전체 보존)
